# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by their '@id', along with their fields/columns
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print(f"  Fields/Columns:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field['@id']}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose major record sets for demonstration. We'll extract all for completeness.
main_recordset_ids = [rs['@id'] for rs in dataset.record_sets]
print('Extracting data from these Record Sets:')
for rid in main_recordset_ids:
    print('  -', rid)

dataframes = {}
for record_set_id in main_recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} columns: {df.columns.tolist()}")

# We'll continue using the first available record set
if main_recordset_ids:
    selected_record_set_id = main_recordset_ids[0]
    print(f"\nSample of record set '{selected_record_set_id}':")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Detect numeric fields in selected record set
df = dataframes[selected_record_set_id]

numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print("Numeric fields available for EDA:", numeric_fields)

# Pick the first numeric field for demo (or insert one known from field overview)
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field '{numeric_field}'")

    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field if present
    categorical_fields = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for col in categorical_fields:
        if col != numeric_field:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean of '{numeric_field}' grouped by '{group_field}':")
        display(grouped_df.head())
else:
    print('No numeric field found for selected record set. Skipping EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot with up to two numeric fields
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter: {numeric_fields[0]} vs. {numeric_fields[1]}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset schema was successfully loaded via the Croissant specification with `mlcroissant`.
* Record sets and their structure were explored using their `@id` references.
* Example EDA and filtering/normalization over numeric fields were demonstrated.
* You may further customize analysis steps depending on your downstream goals (biological insight, annotation exploration, etc.).